In [0]:
CREATE SCHEMA IF NOT EXISTS training_catalog.audit;
USE training_catalog.audit;

In [0]:
-- Create ETL execution log table
CREATE TABLE IF NOT EXISTS etl_run_log (
    RunID BIGINT GENERATED ALWAYS AS IDENTITY,
    Layer STRING,
    TableName STRING,
    RowsRead BIGINT,
    RowsLoaded BIGINT,
    RowsRejected BIGINT,
    Status STRING,
    LoadTimestamp TIMESTAMP
)
USING DELTA;

In [0]:
TRUNCATE TABLE training_catalog.audit.etl_run_log;

In [0]:
-- Log Bronze customer ingestion metrics
INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Bronze','customers_raw',COUNT(*),COUNT(*),0,'SUCCESS',CURRENT_TIMESTAMP()
FROM training_catalog.bronze.customers_raw;

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Bronze','products_raw',COUNT(*),COUNT(*),0,'SUCCESS',CURRENT_TIMESTAMP()
FROM training_catalog.bronze.products_raw;

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Bronze','stores_raw',COUNT(*),COUNT(*),0,'SUCCESS',CURRENT_TIMESTAMP()
FROM training_catalog.bronze.stores_raw;

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Bronze','sales_raw',COUNT(*),COUNT(*),0,'SUCCESS',CURRENT_TIMESTAMP()
FROM training_catalog.bronze.sales_raw;

In [0]:
INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Silver','silver_customers',
(SELECT COUNT(*) FROM training_catalog.bronze.customers_raw),
(SELECT COUNT(*) FROM training_catalog.silver.silver_customers),
(SELECT COUNT(*) FROM training_catalog.reject.rejected_customers),
'SUCCESS',
CURRENT_TIMESTAMP();

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Silver','silver_products',
(SELECT COUNT(*) FROM training_catalog.bronze.products_raw),
(SELECT COUNT(*) FROM training_catalog.silver.silver_products),
(SELECT COUNT(*) FROM training_catalog.reject.rejected_products),
'SUCCESS',
CURRENT_TIMESTAMP();

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Silver','silver_stores',
(SELECT COUNT(*) FROM training_catalog.bronze.stores_raw),
(SELECT COUNT(*) FROM training_catalog.silver.silver_stores),
(SELECT COUNT(*) FROM training_catalog.reject.rejected_stores),
'SUCCESS',
CURRENT_TIMESTAMP();

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Silver','silver_sales',
(SELECT COUNT(*) FROM training_catalog.bronze.sales_raw),
(SELECT COUNT(*) FROM training_catalog.silver.silver_sales),
(SELECT COUNT(*) FROM training_catalog.reject.rejected_sales),
'SUCCESS',
CURRENT_TIMESTAMP();

In [0]:
INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Gold','DimCustomer',
(SELECT COUNT(*) FROM training_catalog.silver.silver_customers),
(SELECT COUNT(*) FROM training_catalog.gold.DimCustomer),
0,'SUCCESS',CURRENT_TIMESTAMP();

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Gold','DimProduct',
(SELECT COUNT(*) FROM training_catalog.silver.silver_products),
(SELECT COUNT(*) FROM training_catalog.gold.DimProduct),
0,'SUCCESS',CURRENT_TIMESTAMP();

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Gold','DimStore',
(SELECT COUNT(*) FROM training_catalog.silver.silver_stores),
(SELECT COUNT(*) FROM training_catalog.gold.DimStore),
0,'SUCCESS',CURRENT_TIMESTAMP();

INSERT INTO etl_run_log
(Layer, TableName, RowsRead, RowsLoaded, RowsRejected, Status, LoadTimestamp)
SELECT 'Gold','FactSales',
(SELECT COUNT(*) FROM training_catalog.silver.silver_sales),
(SELECT COUNT(*) FROM training_catalog.gold.FactSales),
0,'SUCCESS',CURRENT_TIMESTAMP();

In [0]:
SELECT * FROM etl_run_log ORDER BY RunID;